In [7]:
import cv2
import numpy as np
import time, csv, datetime, os
import ctypes

# ==========================================
# USER CONFIGURATION
# ==========================================
ID_CAM = 0

RECT_CONFIG = {'cx': 115, 'cy': 121, 'w': 29, 'h': 27}
ANGLE_CORRECTION = {'x': 1.0, 'y': 1.0, 'taper': 1.0}

OFFSET     = 0.0
EMISSIVITY = 0.95
SCALE      = 3

POLL_INTERVAL    = 0.08   # seconds
RESOLUTION_COEFF = 1.0
RECORD_DURATION  = 4*60*60
CSV_FLUSH_INTERVAL = 500

# --- NOZZLE TRACKER ---
# X position and width are fixed. Only Y tracks within the gate.
# The gate never leaves the neighbourhood of the last known position,
# so the part/bed outside the gate cannot steal the lock.
# Within the gate a TOP-DOWN scan is used: the nozzle is always ABOVE
# the part surface, so it is found first.
ENABLE_TRACKER      = False
TRACKER_CENTER_X    = 160   # fixed horizontal centre (thermal pixels from left)
TRACKER_X_BAND      = 60    # half-width of X search column (thermal pixels)
TRACKER_BOX_W       = 55    # half-width of sampled/display box
TRACKER_BOX_H_ABOVE = 15    # rows above nozzle (dr < 0) in box
TRACKER_BOX_H_BELOW = 100   # rows below nozzle (dr > 0) in box
TRACKER_Y_SMOOTH    = 0.3   # EMA for Y: 1.0 = raw, 0.1 = heavy
TRACKER_SNAP_THR    = 8     # rows — snap directly on large moves
TRACKER_MIN_CONF    = 180.0 # °C — must exceed chamber temp to count as nozzle
TRACKER_GATE_RADIUS = 30    # rows either side of current Y to search
TRACKER_BLOB_SIGMA  = 1.5   # Gaussian blur radius (0 = off)

# ==========================================
# HELPER FUNCTIONS
# ==========================================
def is_shift_pressed():
    try:
        return (ctypes.windll.user32.GetKeyState(0x10) & 0x8000) != 0
    except AttributeError:
        return False

def get_rect_bounds(cfg):
    x1 = cfg['cx'] - cfg['w'] // 2
    y1 = cfg['cy'] - cfg['h'] // 2
    return x1, y1, x1 + cfg['w'], y1 + cfg['h']

def decode_temperatures(thermal):
    lo  = thermal[..., 0].astype(np.int32)
    hi  = thermal[..., 1].astype(np.int32)
    return ((lo + (hi << 8)) / 64.0) - 273.15

def extract_roi_pixels(temps, cfg, taper=1.0, step=1):
    x1, y1, x2, y2 = get_rect_bounds(cfg)
    center_x   = (x1 + x2) / 2.0
    rows_total = y2 - y1
    values = []
    for iy, py in enumerate(range(y1, y2, step)):
        v = min(iy * step / (rows_total - 1) if rows_total > 1 else 0.5, 1.0)
        row_scale = 1.0 + (taper - 1.0) * v
        for px in range(x1, x2, step):
            tx = int(round(center_x + (px - center_x) * row_scale))
            if 0 <= py < temps.shape[0] and 0 <= tx < temps.shape[1]:
                values.append(temps[py, tx])
            else:
                values.append(np.nan)
    return values

def open_camera(index):
    cap = cv2.VideoCapture(index, cv2.CAP_MSMF)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  256)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 384)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc('Y','U','Y','2'))
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
    return cap

def process_frame(cap, cfg, angle_corr, offset, tracker_pos=None, tracker_confident=True):
    ret, frame = cap.read()
    if not ret:
        return None, None, "FAIL"
    buf = frame.ravel()
    if buf.size != 384 * 256 * 2:
        return None, None, "SKIP"

    reshaped = buf.reshape(384, 256, 2)
    visual_half, thermal_half = np.array_split(reshaped, 2)

    temps = decode_temperatures(thermal_half)
    temps = temps / (EMISSIVITY ** 0.25) + offset
    temps = np.rot90(temps, 2)

    visual_bgr = cv2.cvtColor(visual_half, cv2.COLOR_YUV2BGR_YUYV)
    disp       = cv2.resize(visual_bgr, (visual_bgr.shape[1]*SCALE, visual_bgr.shape[0]*SCALE))
    disp_color = cv2.applyColorMap(disp, cv2.COLORMAP_INFERNO)
    disp_color = cv2.rotate(disp_color, cv2.ROTATE_180)

    H, W     = disp_color.shape[:2]
    scale_x  = W / temps.shape[1]
    scale_y  = H / temps.shape[0]

    if ENABLE_TRACKER:
        # Use actual position if locked, otherwise show waiting box at TRACKER_CENTER_X
        tx    = int(round((tracker_pos[0] if tracker_pos else TRACKER_CENTER_X) * scale_x))
        ty    = int(round((tracker_pos[1] if tracker_pos else temps.shape[0] // 2) * scale_y))
        bw    = int(round(TRACKER_BOX_W       * scale_x))
        bh_up = int(round(TRACKER_BOX_H_ABOVE * scale_y))
        bh_dn = int(round(TRACKER_BOX_H_BELOW * scale_y))
        pt1   = (max(0, tx - bw),     max(0, ty - bh_up))
        pt2   = (min(W-1, tx + bw),   min(H-1, ty + bh_dn))
        if tracker_pos is None:
            col = (120, 120, 120)   # grey — not yet acquired
        elif tracker_confident:
            col = (255, 220, 0)     # gold — locked
        else:
            col = (160, 160, 160)   # light grey — low confidence
        cv2.rectangle(disp_color, pt1, pt2, (255,255,255), 3)
        cv2.rectangle(disp_color, pt1, pt2, col, 1)
        cross = max(6, bw // 2)
        cv2.line(disp_color, (tx-cross, ty), (tx+cross, ty), col, 1)
        cv2.line(disp_color, (tx, ty-cross), (tx, ty+cross), col, 1)

        if tracker_pos is not None:
            tcx_t = int(round(tracker_pos[0]))
            tcy_t = int(round(tracker_pos[1]))
            ry1 = max(0, tcy_t - TRACKER_BOX_H_ABOVE)
            ry2 = min(temps.shape[0], tcy_t + TRACKER_BOX_H_BELOW + 1)
            rx1 = max(0, tcx_t - TRACKER_BOX_W)
            rx2 = min(temps.shape[1], tcx_t + TRACKER_BOX_W + 1)
            box_region   = temps[ry1:ry2, rx1:rx2]
            hot_val      = float(np.nanmax(box_region)) if box_region.size > 0 else float('nan')
            below_region = temps[min(temps.shape[0], tcy_t+1):ry2, rx1:rx2]
            below_val    = float(np.nanmax(below_region)) if below_region.size > 0 else float('nan')
            conf_str     = '' if tracker_confident else ' (low conf)'
            lx = min(tx + bw + 4, W - 140)
            ly = max(ty - bh_up - 6, 16)
            for label, dy in [(f"Nozzle: {hot_val:.1f}C{conf_str}", 0),
                              (f"Below:  {below_val:.1f}C", 18)]:
                cv2.putText(disp_color, label, (lx, ly+dy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
                cv2.putText(disp_color, label, (lx, ly+dy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)

    else:
        # Fixed ROI rectangle — move with arrow keys, resize with Shift+arrows
        x1r = int((cfg['cx'] - cfg['w']//2) * scale_x)
        y1r = int((cfg['cy'] - cfg['h']//2) * scale_y)
        x2r = int((cfg['cx'] + cfg['w']//2) * scale_x)
        y2r = int((cfg['cy'] + cfg['h']//2) * scale_y)
        cv2.rectangle(disp_color, (x1r, y1r), (x2r, y2r), (255, 255, 255), 3)
        cv2.rectangle(disp_color, (x1r, y1r), (x2r, y2r), (0, 255, 0), 1)
        cv2.putText(disp_color,
                    f"ROI cx={cfg['cx']} cy={cfg['cy']} w={cfg['w']} h={cfg['h']}",
                    (x1r, max(y1r - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)

    step     = max(1, round(1.0/RESOLUTION_COEFF)) if RESOLUTION_COEFF < 1.0 else 1
    roi_vals = extract_roi_pixels(temps, cfg, taper=angle_corr.get('taper',1.0), step=step)
    finite   = [v for v in roi_vals if np.isfinite(v)]
    avg_temp = np.mean(finite) if finite else 0.0
    cv2.putText(disp_color,
                f"Off:{offset:+.1f}C | Avg:{avg_temp:.1f}C | Px:{len(roi_vals)} (step={step})",
                (10, H-20), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,255,0), 2)

    return disp_color, temps, "OK"


# ==========================================
# TRACKER: top-down scan within fixed X band
# ==========================================
def _tracker_scan(temps, cur_y, x1, x2, y_lo, y_hi):
    """
    Scan rows top-to-bottom within [y_lo, y_hi).
    Return (hx, hy, peak_val) for the FIRST row whose mean across [x1,x2]
    exceeds TRACKER_MIN_CONF.  The nozzle is always above the part so it
    is found before the hot part surface.
    Returns None if nothing qualifies.
    """
    y_lo = max(0, y_lo); y_hi = min(temps.shape[0], y_hi)
    if y_lo >= y_hi:
        return None
    band      = temps[y_lo:y_hi, x1:x2]
    row_means = np.nanmean(band, axis=1)
    for ry_local in range(band.shape[0]):          # top → bottom
        rv = row_means[ry_local]
        if TRACKER_MIN_CONF <= rv <= TRACKER_MAX_TEMP:
            ry  = y_lo + ry_local
            hx  = x1 + int(np.nanargmax(band[ry_local]))
            return hx, ry, float(rv)
    return None


# ==========================================
# MAIN LOOP
# ==========================================
def main():
    global RECT_CONFIG, ANGLE_CORRECTION, OFFSET

    recording           = False
    csv_rows            = []
    csv_file            = None
    csv_writer          = None
    csv_frame_count     = 0
    last_csv_time       = 0
    csv_interval        = POLL_INTERVAL
    video_writer        = None
    record_video        = False
    csv_start           = None
    vid_start           = None
    csv_path            = None
    tracker_pos         = None    # (cx, cy) in thermal pixels — Y only moves
    tracker_confident   = False
    tracker_low_conf    = 0

    output_folder = "Data Recordings"
    os.makedirs(output_folder, exist_ok=True)

    cap = open_camera(ID_CAM)
    if not cap.isOpened():
        print("ERROR: Could not open camera."); return

    timer_info = f"  Auto-stop after {RECORD_DURATION}s" if RECORD_DURATION else "  Manual stop [T]/[S]"
    print(f"System Started.{timer_info}")
    print("Controls: [Q]uit | [R]CSV | [V]Video | [T]SaveCSV | [S]SaveVid | [C]Clear tracker")
    print(f"  Poll={POLL_INTERVAL}s  TRACKER_CENTER_X={TRACKER_CENTER_X}  TRACKER_X_BAND={TRACKER_X_BAND}")

    step      = max(1, round(1.0/RESOLUTION_COEFF)) if RESOLUTION_COEFF < 1.0 else 1
    last_poll = 0.0

    # Fixed X band in thermal pixel space
    x1_band = max(0, TRACKER_CENTER_X - TRACKER_X_BAND)
    x2_band = 256   # will be clipped to actual image width on first frame

    try:
        while True:
            now_poll = time.time()
            if now_poll - last_poll < POLL_INTERVAL:
                time.sleep(0.001); continue
            last_poll = now_poll

            view, temps, status = process_frame(
                cap, RECT_CONFIG, ANGLE_CORRECTION, OFFSET, tracker_pos, tracker_confident)
            if status == "SKIP": continue
            if status == "FAIL": break

            # Clip X band to actual image width now that we have temps
            x2_band = min(temps.shape[1], TRACKER_CENTER_X + TRACKER_X_BAND + 1)

            # ---- Tracker update ----
            if ENABLE_TRACKER and temps is not None:
                if tracker_pos is None:
                    # Initial acquisition: find hottest pixel in X band within
                    # [TRACKER_MIN_CONF, TRACKER_MAX_TEMP].  The nozzle is always
                    # the hottest thing in that window — IR lamps are excluded by
                    # TRACKER_MAX_TEMP, the chamber/part are excluded by TRACKER_MIN_CONF.
                    band = temps[:, x1_band:x2_band].copy()
                    # Zero out pixels outside the valid temperature window
                    valid = (band >= TRACKER_MIN_CONF) & (band <= TRACKER_MAX_TEMP)
                    if np.any(valid):
                        masked = np.where(valid, band, -np.inf)
                        flat   = int(np.argmax(masked))
                        hy     = flat // band.shape[1]
                        hx     = x1_band + flat % band.shape[1]
                        pv     = float(band[hy, flat % band.shape[1]])
                        tracker_pos       = (float(TRACKER_CENTER_X), float(hy))
                        tracker_confident = True
                        tracker_low_conf  = 0
                        print(f'Tracker acquired at row {hy}  ({pv:.1f} C)')
                else:
                    cur_y = tracker_pos[1]
                    # Gate search: top-down within ±GATE rows of current Y
                    result = _tracker_scan(temps, cur_y, x1_band, x2_band,
                                           int(cur_y) - TRACKER_GATE_RADIUS,
                                           int(cur_y) + TRACKER_GATE_RADIUS + 1)
                    if result:
                        _, hy, pv         = result
                        dy                = abs(hy - cur_y)
                        new_y             = float(hy) if dy > TRACKER_SNAP_THR \
                                            else TRACKER_Y_SMOOTH*hy + (1-TRACKER_Y_SMOOTH)*cur_y
                        tracker_pos       = (float(TRACKER_CENTER_X), new_y)
                        tracker_confident = True
                        tracker_low_conf  = 0
                    else:
                        # Hold last position — nozzle obscured or between moves
                        tracker_confident = False
                        tracker_low_conf += 1

            # ---- Auto-stop ----
            csv_remaining = vid_remaining = None
            if RECORD_DURATION:
                if recording and csv_start is not None:
                    csv_remaining = RECORD_DURATION - (time.time() - csv_start)
                    if csv_remaining <= 0:
                        recording = False; csv_start = None
                        if csv_rows: csv_writer.writerows(csv_rows); csv_rows = []
                        csv_file.close(); csv_file = csv_writer = None
                        print(f'Auto-stop: CSV saved → {csv_path} ({csv_frame_count} frames)')
                if record_video and vid_start is not None:
                    vid_remaining = RECORD_DURATION - (time.time() - vid_start)
                    if vid_remaining <= 0:
                        record_video = False; vid_start = None
                        if video_writer: video_writer.release(); video_writer = None
                        print("Auto-stop: video saved.")

            # ---- UI bar ----
            ui_h  = 100
            ui_bar = np.zeros((ui_h, view.shape[1], 3), dtype=np.uint8)
            rec_col = (0,0,255) if recording    else (80,80,80)
            vid_col = (0,0,255) if record_video else (80,80,80)
            rec_lbl = f"CSV REC  {csv_frame_count}fr" if recording else "CSV RECORDING"
            cv2.putText(ui_bar, rec_lbl,          (20,30),  cv2.FONT_HERSHEY_SIMPLEX, 0.6, rec_col, 2)
            cv2.putText(ui_bar, "VIDEO RECORDING", (280,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, vid_col, 2)
            if ENABLE_TRACKER:
                conf_col = (0,220,80) if tracker_confident else (80,80,160)
                if tracker_pos is None:
                    conf_lbl = "NOZZLE: searching..."
                elif tracker_confident:
                    conf_lbl = f"NOZZLE LOCKED  cy={int(tracker_pos[1])}"
                else:
                    conf_lbl = f"NOZZLE: low conf ({tracker_low_conf})"
                cv2.putText(ui_bar, conf_lbl, (20,55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, conf_col, 1)
            ts_disp = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
            cv2.putText(ui_bar, ts_disp, (ui_bar.shape[1]-220,30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            if csv_remaining and csv_remaining > 0:
                cv2.putText(ui_bar, f"CSV stop in {csv_remaining:.0f}s", (280,55), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,165,255), 1)
            cv2.putText(ui_bar, "[R]CSV [V]Vid [T]SaveCSV [S]SaveVid [Q]uit [C]Reset tracker",
                        (20,75), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)
            cv2.putText(ui_bar, "[+/-]:Offset",
                        (20,92), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (200,200,200), 1)

            final = np.vstack([ui_bar, view])
            if recording and int(time.time()*2)%2 == 0:
                cv2.circle(final, (30, ui_h+30), 10, (0,0,255), -1)
            cv2.imshow("Thermal Scanner", final)

            # ---- Video write ----
            if record_video:
                h, w = final.shape[:2]
                if video_writer is None:
                    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                    fn = os.path.join(output_folder, f"thermal_video_{ts}.avi")
                    video_writer = cv2.VideoWriter(fn, cv2.VideoWriter_fourcc(*'MJPG'), 20.0, (w,h))
                video_writer.write(final)

            # ---- CSV write ----
            if recording:
                now = time.time()
                if now - last_csv_time >= csv_interval:
                    if ENABLE_TRACKER and tracker_pos is not None:
                        tcx = int(round(tracker_pos[0]))
                        tcy = int(round(tracker_pos[1]))
                        pixels = []
                        for ry in range(tcy - TRACKER_BOX_H_ABOVE, tcy + TRACKER_BOX_H_BELOW + 1):
                            for rx in range(tcx - TRACKER_BOX_W, tcx + TRACKER_BOX_W + 1):
                                if 0 <= ry < temps.shape[0] and 0 <= rx < temps.shape[1]:
                                    pixels.append(temps[ry, rx])
                                else:
                                    pixels.append(np.nan)
                    else:
                        # Fixed ROI — sample a centred grid matching the header
                        tcx = RECT_CONFIG['cx']
                        tcy = RECT_CONFIG['cy']
                        bha = RECT_CONFIG['h'] // 2
                        bhb = RECT_CONFIG['h'] // 2
                        bw  = RECT_CONFIG['w'] // 2
                        pixels = []
                        for ry in range(tcy - bha, tcy + bhb + 1):
                            for rx in range(tcx - bw, tcx + bw + 1):
                                if 0 <= ry < temps.shape[0] and 0 <= rx < temps.shape[1]:
                                    pixels.append(temps[ry, rx])
                                else:
                                    pixels.append(np.nan)

                    row = [f"{now:.3f}".replace('.',','),
                           f"{now-csv_start:.3f}".replace('.',','),
                           f"{tcx}", f"{tcy}"]
                    for val in pixels:
                        row.append(f"{val:.3f}".replace('.',',') if np.isfinite(val) else "")
                    csv_rows.append(row)
                    csv_frame_count += 1
                    last_csv_time = now
                    if csv_frame_count % CSV_FLUSH_INTERVAL == 0:
                        csv_writer.writerows(csv_rows); csv_file.flush(); csv_rows = []

            # ---- Keyboard ----
            full_key = cv2.waitKeyEx(1)
            if full_key != -1:
                char_key = full_key & 0xFF
                if full_key in [2490368, 0x260000, 65362]:
                    RECT_CONFIG['h' if is_shift_pressed() else 'cy'] -= 2
                elif full_key in [2621440, 0x280000, 65364]:
                    RECT_CONFIG['h' if is_shift_pressed() else 'cy'] += 2
                elif full_key in [2424832, 0x250000, 65361]:
                    RECT_CONFIG['w' if is_shift_pressed() else 'cx'] -= 2
                elif full_key in [2555904, 0x270000, 65363]:
                    RECT_CONFIG['w' if is_shift_pressed() else 'cx'] += 2
                elif char_key in (ord('q'), 27):
                    break
                elif char_key == ord('c'):
                    tracker_pos = None; tracker_confident = False; tracker_low_conf = 0
                    print('Tracker reset — will re-acquire on next frame')
                elif char_key in (ord('+'), 43): OFFSET += 0.1
                elif char_key in (ord('-'), 45): OFFSET -= 0.1
                elif char_key in (ord('0'), 48): ANGLE_CORRECTION['taper'] += 0.05
                elif char_key in (ord('9'), 57): ANGLE_CORRECTION['taper'] -= 0.05
                elif char_key == ord('r'):
                    if not recording:
                        ts_fn      = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
                        csv_path   = os.path.join(output_folder, f"thermal_data_{ts_fn}.csv")
                        csv_file   = open(csv_path, 'w', newline='')
                        csv_writer = csv.writer(csv_file, delimiter=';')
                        bha = TRACKER_BOX_H_ABOVE if ENABLE_TRACKER else RECT_CONFIG['h']//2
                        bhb = TRACKER_BOX_H_BELOW if ENABLE_TRACKER else RECT_CONFIG['h']//2
                        bw  = TRACKER_BOX_W        if ENABLE_TRACKER else RECT_CONFIG['w']//2
                        hdr = ['timestamp','elapsed_s','nozzle_cx','nozzle_cy']
                        for dr in range(-bha, bhb+1):
                            for dc in range(-bw, bw+1):
                                hdr.append(f'px_dr{dr:+d}_dc{dc:+d}')
                        csv_writer.writerow(hdr); csv_file.flush()
                        csv_rows = []; csv_frame_count = 0
                        recording = True; csv_start = time.time()
                        print(f'CSV recording → {csv_path}')
                elif char_key == ord('v'):
                    if not record_video:
                        record_video = True; vid_start = time.time()
                elif char_key == ord('t'):
                    if recording:
                        recording = False; csv_start = None
                        if csv_rows: csv_writer.writerows(csv_rows); csv_rows = []
                        csv_file.close(); csv_file = csv_writer = None
                        print(f'CSV saved → {csv_path} ({csv_frame_count} frames)')
                elif char_key == ord('s'):
                    if record_video:
                        record_video = False; vid_start = None
                        if video_writer: video_writer.release(); video_writer = None

    finally:
        cap.release()
        if video_writer: video_writer.release()
        if csv_file:
            if csv_rows: csv_writer.writerows(csv_rows)
            csv_file.close()
            print(f'CSV closed on exit → {csv_path} ({csv_frame_count} frames)')
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


System Started.  Auto-stop after 14400s
Controls: [Q]uit | [R]CSV | [V]Video | [T]SaveCSV | [S]SaveVid | [C]Clear tracker
  Poll=0.08s  TRACKER_CENTER_X=160  TRACKER_X_BAND=60
CSV recording → Data Recordings\thermal_data_20260323_151029.csv
CSV saved → Data Recordings\thermal_data_20260323_151029.csv (12893 frames)
CSV recording → Data Recordings\thermal_data_20260323_153235.csv
CSV saved → Data Recordings\thermal_data_20260323_153235.csv (55916 frames)
